# CIFAR-10 CNN — Applied Version

This notebook is based directly on the supplied CNN/CIFAR-10 notebook. It keeps the same dataset and CNN workflow, while fixing the label/loss consistency and adding a clean evaluation section.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

train_labels_cat = to_categorical(train_labels, 10)
test_labels_cat = to_categorical(test_labels, 10)

print("Train:", train_images.shape, train_labels_cat.shape)
print("Test :", test_images.shape, test_labels_cat.shape)

In [ ]:
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

plt.figure(figsize=(10, 8))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(train_images[i])
    plt.xticks([]); plt.yticks([])
    plt.title(class_names[train_labels[i][0]])
plt.tight_layout()
plt.show()

## CNN model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_images,
    train_labels_cat,
    epochs=10,
    validation_split=0.1,
    batch_size=128,
    verbose=1
)

In [ ]:
test_loss, test_acc = model.evaluate(
    test_images, test_labels_cat, verbose=2
)
print("Test accuracy:", test_acc)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CIFAR-10 CNN Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CIFAR-10 CNN Loss")
plt.legend()
plt.show()

In [ ]:
pred = np.argmax(model.predict(test_images, verbose=0), axis=1)
true = test_labels.flatten()

print(classification_report(true, pred, target_names=class_names))

cm = confusion_matrix(true, pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm)
plt.colorbar()
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CIFAR-10 Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
model.save("cifar10_cnn_applied.keras")
print("Saved: cifar10_cnn_applied.keras")